In [1]:
import os
import subprocess
import json
from langchain_google_genai import GoogleGenerativeAI

llm = GoogleGenerativeAI(model="gemini-3.6-flash", google_api_key="API_KEY")

In [2]:

md_folder = r"F:\FAQ"

for filename in os.listdir(md_folder):
    if filename.endswith(".md"):
        filepath = os.path.join(md_folder, filename)
        subprocess.run([
            "python", "run_pageindex.py",
            "--md_path", filepath,
            "--model", "gemini/gemini-2.5-flash"
        ])

In [3]:
with open(r"F:\FAQ\PageIndex\results\airline_knowledge_base_structure.json", "r", encoding="utf-8") as f:
    tree = json.load(f)

print(json.dumps(tree, indent=2)[:2000])

{
  "doc_name": "airline_knowledge_base",
  "line_count": 510,
  "structure": [
    {
      "title": "Merged Files List",
      "node_id": "0001",
      "line_num": 1
    },
    {
      "title": "1. airport_services.md",
      "node_id": "0002",
      "line_num": 29
    },
    {
      "title": "2. baggage_policy.md",
      "node_id": "0003",
      "line_num": 46
    },
    {
      "title": "3. boarding_policy.md",
      "node_id": "0004",
      "line_num": 68
    },
    {
      "title": "4. booking_policy.md",
      "node_id": "0005",
      "line_num": 82
    },
    {
      "title": "5. checkin_policy.md",
      "node_id": "0006",
      "line_num": 105
    },
    {
      "title": "6. coupon_policy.md",
      "node_id": "0007",
      "line_num": 123
    },
    {
      "title": "7. customer_support_policy.md",
      "node_id": "0008",
      "line_num": 139
    },
    {
      "title": "8. delay_compensation_policy.md",
      "node_id": "0009",
      "line_num": 157
    },
    {
      "tit

In [4]:
sections_text = "\n".join(
    f"- node_id: {node['node_id']}, title: {node['title']}"
    for node in tree["structure"]
)
print(sections_text)

- node_id: 0001, title: Merged Files List
- node_id: 0002, title: 1. airport_services.md
- node_id: 0003, title: 2. baggage_policy.md
- node_id: 0004, title: 3. boarding_policy.md
- node_id: 0005, title: 4. booking_policy.md
- node_id: 0006, title: 5. checkin_policy.md
- node_id: 0007, title: 6. coupon_policy.md
- node_id: 0008, title: 7. customer_support_policy.md
- node_id: 0009, title: 8. delay_compensation_policy.md
- node_id: 0010, title: 9. emergency_contact.md
- node_id: 0011, title: 10. flight_cancellation_policy.md
- node_id: 0012, title: 11. frequent_flyer_rewards.md
- node_id: 0013, title: 12. insurance_policy.md
- node_id: 0014, title: 13. loyalt_program.md
- node_id: 0015, title: 14. luggage_damage_policy.md
- node_id: 0016, title: 15. meal_policy.md
- node_id: 0017, title: 16. payment_policy.md
- node_id: 0018, title: 17. pet_travel_policy.md
- node_id: 0019, title: 18. promo_offer_policy.md
- node_id: 0020, title: 19. refund_policy.md
- node_id: 0021, title: 20. reschedu

In [5]:
question = "What is the refund policy?"

routing_prompt = f"""
You are given a list of document sections:

{sections_text}

Based on the section titles, which single node_id is most relevant to answer this question?
Question: {question}

Respond with ONLY the node_id, nothing else.
"""

chosen_node_id = llm.invoke(routing_prompt).strip()
print("Chosen node:", chosen_node_id)

c:\Users\jesse\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Chosen node: 0020


In [6]:
with open(r"F:\FAQ\airline_knowledge_base.md", "r", encoding="utf-8") as f:
    md_lines = f.readlines()

In [7]:
def get_node_text(tree, node_id, md_lines):
    nodes = tree["structure"]
    
    # find the index of this node in the list
    idx = next(i for i, n in enumerate(nodes) if n["node_id"] == node_id)
    
    start_line = nodes[idx]["line_num"] - 1  # convert to 0-indexed
    
    # if there's a next node, stop right before it; otherwise read to end of file
    if idx + 1 < len(nodes):
        end_line = nodes[idx + 1]["line_num"] - 1
    else:
        end_line = len(md_lines)
    
    section_text = "".join(md_lines[start_line:end_line])
    return section_text

In [8]:
node_text = get_node_text(tree, chosen_node_id, md_lines)
print(node_text)


```md
# Refund Policy

Last Updated: August 2026

## General Refunds
Customers may request a refund within 7 days of purchase if the service has not been fully utilized.

## Cancellation Before Service
If a booking is cancelled more than 48 hours before departure, customers receive a full refund.

## Cancellation Within 48 Hours
Customers receive an 80% refund.

## Cancellation Within 12 Hours
Only taxes and government charges are refundable.

## Non-refundable Cases
- Promotional bookings
- Gift cards
- Loyalty point purchases
- No-show bookings

Refunds are processed within 5–7 business days.
```

## 20. rescheduling_policy.md



In [9]:
answer_prompt = f"""
Answer the question using only the following section of the airline knowledge base:

{node_text}

Question: {question}
"""

final_answer = llm.invoke(answer_prompt)
print(final_answer)

c:\Users\jesse\anaconda3\Lib\site-packages\langchain_google_genai\chat_models.py:3120: UserWarning: Model 'models/gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided section, the refund policy is as follows:

* **General Refunds:** Customers can request a refund within 7 days of purchase if the service has not been fully utilized.
* **Cancellation Timelines:**
  * **More than 48 hours before departure:** Full refund.
  * **Within 48 hours before departure:** 80% refund.
  * **Within 12 hours before departure:** Only taxes and government charges are refundable.
* **Non-refundable Cases:** 
  * Promotional bookings
  * Gift cards
  * Loyalty point purchases
  * No-show bookings
* **Processing Time:** Refunds are processed within 5–7 business days.
